In [2]:
from qdrant_client import QdrantClient
from sklearn.model_selection import train_test_split
import os, mlflow, optuna, numpy as np
from dotenv import load_dotenv
import trainer, torch
from dataset import CarOfferDataset
from torch.utils.data import DataLoader
from models import OfferCompressor
import pandas as pd

In [3]:
COLLECTION_NAME = "cars"

In [4]:
experiment_name = "embeddings-compressor-hyperparameter-tuning"

mlflow.set_experiment(experiment_name)

2026/09/11 22:33:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/11 22:33:51 INFO mlflow.store.db.utils: Updating database tables
2026/09/11 22:33:53 INFO mlflow.tracking.fluent: Experiment with name 'embeddings-compressor-hyperparameter-tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='/kaggle/working/mlruns/1', creation_time=1789166033422, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789166033422, lifecycle_stage='active', name='embeddings-compressor-hyperparameter-tuning', tags={}, trace_location=None, workspace='default'>

In [5]:
num_clusters = [4, 6, 8]
target_dimensions = [64, 128]
K_NEIGHBOURS = 5
NUM_HEADS = 4
NUM_EPOCHS = 100
BATCH_SIZE = 32
INPUT_DIMENSION = 768

In [6]:
load_dotenv('/kaggle/input/datasets/enol00/secret/.env')

client = QdrantClient(
    api_key=os.environ['QDRANT_API_KEY'],
    url = os.environ['QDRANT_URL'],
)

In [7]:
groups_result = client.query_points_groups(
    collection_name=COLLECTION_NAME,
    limit=4400,
    group_by='url',
    with_vectors=True,
    with_payload=["brand", "model", "url"]
)

groups = groups_result.groups

records = [group.hits[0] for group in groups if group.hits]

grouped = {item.payload['url']: item.vector for item in records}

train, test = train_test_split(
    records,
    shuffle=True,
    test_size=0.2,
    random_state=42
)

In [8]:
train_embeddings = np.stack([item.vector for item in train])
train_ids = np.array([item.payload['url'] for item in train], dtype="U64")
train_metadata = np.array([item.payload['brand'] + item.payload["model"] for item in train], dtype="U64")


test_embeddings = np.stack([item.vector for item in test])
test_ids = np.array([item.payload['url'] for item in test], dtype="U64")
test_metadata = np.array([item.payload['brand'] + item.payload["model"] for item in test], dtype="U64")

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def collate_offer_batch(batch):
    max_photos = max(item[0].size(0) for item in batch)
    embed_dim = batch[0][0].size(1)
    batch_size = len(batch)

    padded_x = torch.zeros(batch_size, max_photos, embed_dim)
    padding_mask = torch.ones(batch_size, max_photos, dtype=torch.bool)

    urls = []
    metadata = []

    for i, (embeds, url, meta) in enumerate(batch):
        num_photos = embeds.size(0)
        padded_x[i, :num_photos] = embeds
        padding_mask[i, :num_photos] = False
        urls.append(url)
        metadata.append(meta)

    return padded_x, padding_mask, urls, metadata

In [10]:
train_dataset = CarOfferDataset(train_embeddings, train_ids, train_metadata)
test_dataset = CarOfferDataset(test_embeddings, test_ids, test_metadata)

train_loader = DataLoader(train_dataset, 
                          batch_size=BATCH_SIZE, 
                          shuffle=True, 
                          num_workers=4,
                          pin_memory=True,                          
                          collate_fn=collate_offer_batch
)

test_loader = DataLoader(test_dataset, 
                         batch_size=BATCH_SIZE, 
                         shuffle=False, 
                         num_workers=4,
                         pin_memory=True,
                         collate_fn=collate_offer_batch
)

In [11]:
def objective(trial):
    with mlflow.start_run(nested=True):

        clusters = trial.suggest_categorical("num_clusters", num_clusters)
        target_dimension = trial.suggest_categorical("target_dimension", target_dimensions)
        lambda_cluster = trial.suggest_float("lambda_cluster", 0.1, 1.0)
        
        compressor = OfferCompressor(
            input_dim=INPUT_DIMENSION,
            target_dim=target_dimension,
            num_clusters=clusters,
            num_heads=NUM_HEADS,
            batch_size=BATCH_SIZE
        ).to(device)

        trainer.train_compressor(
            model=compressor,
            epochs=NUM_EPOCHS,
            train_loader=train_loader,
            lambda_cluster=lambda_cluster,
            device=device
        )

        evaluations = trainer.evaluate_compressor(
            model=compressor,
            val_loader=test_loader,
            k=K_NEIGHBOURS
        )

        mlflow.log_params(trial.params)

        mlflow.log_params({
            "number_of_heads": NUM_HEADS,
            "epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "k_neighbours": K_NEIGHBOURS
        })

        mlflow.log_metrics(evaluations)

        model_filename = f"model_trial_{trial.number}.pt"
        torch.save(compressor.state_dict(), model_filename)
        mlflow.log_artifact(model_filename, artifact_path="checkpoints")
        os.remove(model_filename)

        return evaluations['normalized_dcg']

In [12]:
with mlflow.start_run(run_name="optuna_study"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)

[I 2026-09-11 22:33:58,359] A new study created in memory with name: no-name-7fc0a667-0cfb-4b78-9156-5e99e6868bcf
[I 2026-09-11 22:35:35,140] Trial 0 finished with value: 0.8731101751327515 and parameters: {'num_clusters': 8, 'target_dimension': 128, 'lambda_cluster': 0.6540954716377422}. Best is trial 0 with value: 0.8731101751327515.
[I 2026-09-11 22:37:10,160] Trial 1 finished with value: 0.8303347229957581 and parameters: {'num_clusters': 6, 'target_dimension': 64, 'lambda_cluster': 0.1640816319118134}. Best is trial 0 with value: 0.8731101751327515.
[I 2026-09-11 22:38:44,643] Trial 2 finished with value: 0.8314489126205444 and parameters: {'num_clusters': 6, 'target_dimension': 128, 'lambda_cluster': 0.5414410734467885}. Best is trial 0 with value: 0.8731101751327515.
[I 2026-09-11 22:40:19,972] Trial 3 finished with value: 0.8730215430259705 and parameters: {'num_clusters': 8, 'target_dimension': 64, 'lambda_cluster': 0.9420027823162817}. Best is trial 0 with value: 0.8731101751

In [15]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

experiment = mlflow.get_experiment_by_name(experiment_name)

df_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

param_cols = [c for c in df_runs.columns if c.startswith("params.")]
metric_cols = [c for c in df_runs.columns if c.startswith("metrics.")]

selected_cols = ["run_id", "status", "start_time"] + param_cols + metric_cols
df_clean = df_runs[selected_cols]

sort_by_col = "metrics.normalized_dcg" if "metrics.normalized_dcg" in df_clean.columns else metric_cols[0]
df_clean = df_clean.sort_values(by=sort_by_col, ascending=False)

display(df_clean.head(10))
df_clean.to_csv("mlflow_results_extracted.csv", index=False)

,run_id,status,start_time,params.number_of_heads,params.lambda_cluster,params.k_neighbours,params.num_clusters,params.batch_size,params.epochs,params.target_dimension,metrics.precision_k,metrics.mrr,metrics.map,metrics.recall_k,metrics.hit,metrics.normalized_dcg
5,30e8a6bb64db4a4a9331acb6adafcef2,FINISHED,2026-09-11 22:56:41.685000+00:00,4,0.7805207525485832,5,8,32,100,64,0.874333,0.934583,0.704173,0.444558,1.000000,0.873463
19,a3f6a5d461054b19801a4bd4a04a12c0,FINISHED,2026-09-11 22:33:58.363000+00:00,4,0.6540954716377422,5,8,32,100,128,0.873833,0.934583,0.699938,0.444201,1.000000,0.873110
4,53faa7b5c1c449a7b0f76924ec4d3540,FINISHED,2026-09-11 22:58:22.602000+00:00,4,0.6562221832430527,5,8,32,100,128,0.873833,0.934213,0.697625,0.444201,0.999167,0.873022
6,2314146d477c4a25b550e9604a2989bd,FINISHED,2026-09-11 22:55:00.703000+00:00,4,0.9956458856504913,5,8,32,100,128,0.873833,0.934179,0.704173,0.444201,0.999167,0.873022
16,4d9be76bf9b643ec8e6d3439de75cff2,FINISHED,2026-09-11 22:38:44.648000+00:00,4,0.9420027823162817,5,8,32,100,64,0.873833,0.934187,0.695420,0.444201,0.999167,0.873022
7,2e5353a0ef0c4c4ca6b1d5a658eea56e,FINISHED,2026-09-11 22:53:19.612000+00:00,4,0.7313131779128634,5,8,32,100,128,0.873833,0.934213,0.701657,0.444201,0.999167,0.873022
2,cbc1db0ab8da438c82ce9a237667f65b,FINISHED,2026-09-11 23:01:50.520000+00:00,4,0.5913848672454798,5,8,32,100,128,0.873667,0.934199,0.699872,0.444082,0.999167,0.872912
9,ef88c1e969e94dfea60d3e71d8dec994,FINISHED,2026-09-11 22:49:56.643000+00:00,4,0.764663783620978,5,8,32,100,128,0.873500,0.934583,0.697725,0.443963,1.000000,0.872891
8,e25f53cf54d344648996ee738b932580,FINISHED,2026-09-11 22:51:35.466000+00:00,4,0.9725462318614733,5,8,32,100,128,0.873167,0.934583,0.698953,0.443725,1.000000,0.872596
0,9f30124d179e47e2a12a828dc65902a0,FINISHED,2026-09-11 23:05:19.826000+00:00,4,0.5062586136937727,5,8,32,100,64,0.873000,0.934250,0.698586,0.443606,0.999167,0.872475
